# Minimum-path, energetic, self-reaction, and stability screening

Retain the minimum predicted aqueous activation free energy for each reactant pair, then apply the reaction free-energy and barrier limits. The final reactant-level exclusion combines five self-reactivity identifiers with six structures manually judged potentially unstable. A pair is removed if either reactant appears in either list. The obsolete functional-group hard filter is not part of the current workflow.

The minimum-path and energetic stages read or write the saved intermediate files under `Data/Predict_DI`. The final exclusion code reproduces the curated lists used in `4_DI_discover.ipynb`.

In [1]:
from pathlib import Path
import os
import sys
from IPython.display import display

start = Path.cwd().resolve()
REPO_ROOT = next((p for p in (start, *start.parents)
                  if (p / "Code" / "model_train.py").is_file()), None)
if REPO_ROOT is None:
    raise FileNotFoundError("Start Jupyter within the CycloAddGenerator repository.")
for folder in (REPO_ROOT, REPO_ROOT / "notebooks",
               REPO_ROOT / "notebooks/04_discovery", REPO_ROOT / "notebooks/05_mechanism"):
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))
os.chdir(REPO_ROOT)


In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from matplotlib import pyplot as plt
from _workflow_paths import *


c:\Users\Jackie\anaconda3\envs\main_py3_12\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Minimum-path reduction
This step reads the large external prediction files. The following section can also be started from existing `1_deltaG.csv` / `2_deltaGa.csv` results.

In [3]:
prediction_csvs = sorted(PRED_DIR.glob("Diene_*.csv"))
if not prediction_csvs:
    raise FileNotFoundError(PRED_DIR)
ONE_PATH_DIR.mkdir(parents=True, exist_ok=True)
for csv_path in tqdm(prediction_csvs):
    frame = pd.read_csv(csv_path).dropna().reset_index()
    chosen = frame.groupby(["Diene_Index", "Ene_Index"], sort=False)["deltaGa(Solvent)_p"].idxmin()
    frame.loc[chosen].to_csv(ONE_PATH_DIR / csv_path.name, index=False)


100%|██████████| 720/720 [01:32<00:00,  7.76it/s]


In [4]:
csvs = sorted(ONE_PATH_DIR.glob("Diene_Index_*.csv"))
if not csvs:
    raise FileNotFoundError(ONE_PATH_DIR)
frames = []
for csv_path in tqdm(csvs):
    frame = pd.read_csv(csv_path)
    frames.append(frame.loc[frame["deltaG_p"] < -5])
screened = pd.concat(frames, ignore_index=True)
SCREEN_DIR.mkdir(parents=True, exist_ok=True)
screened.to_csv(SCREEN_DIR / "1_deltaG.csv", index=False)


100%|██████████| 718/718 [00:07<00:00, 102.47it/s]


In [ ]:
screened = pd.read_csv(require_file(SCREEN_DIR / "1_deltaG.csv"))
screened = screened.loc[screened["deltaGa(Solvent)_p"] < 24].copy()
screened.to_csv(SCREEN_DIR / "2_deltaGa.csv", index=False)


## Self-reactivityexclusions


In [ ]:
SELF_REACTION_IDS = {12293, 13781, 14107, 15681, 17705, 17935, 13787, 16502, 13758, 17291, 13569}

screened = pd.read_csv(require_file(SCREEN_DIR / "2_deltaGa.csv"))
self_reaction_mask = screened["Diene_Index"].isin(SELF_REACTION_IDS) | screened["Ene_Index"].isin(SELF_REACTION_IDS)
after_self_reaction = screened.loc[~self_reaction_mask].copy()

banned_ids = sorted(SELF_REACTION_IDS | MANUAL_UNSTABLE_IDS)
output_path = SCREEN_DIR / "3_DropDimer.csv"
after_self_reaction.to_csv(output_path, index=False)

screening_summary = pd.DataFrame({
    "Stage": ["After energetic screening", "After self-reactivity exclusion", "After manual structural-stability exclusion"],
    "Reaction pairs": [len(screened), len(after_self_reaction), len(after_self_reaction)],
    "Diene IDs": [screened["Diene_Index"].nunique(), after_self_reaction["Diene_Index"].nunique(), after_self_reaction["Diene_Index"].nunique()],
    "Ene IDs": [screened["Ene_Index"].nunique(), after_self_reaction["Ene_Index"].nunique(), after_self_reaction["Ene_Index"].nunique()],
})
display(screening_summary)
print(f"Saved {len(after_self_reaction):,} reaction pairs to {output_path}")